In [58]:
import os
import json
import math
import random
import numpy as np
from collections import Counter
from dataclasses import dataclass
from typing import List, Dict, Tuple, Any
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import hf_hub_download, login

# Configurations and hyperparameters

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_TOKEN = os.environ.get("HF_TOKEN", None)
if HF_TOKEN:
    login(token=HF_TOKEN)

# Paths for Phase 1 Checkpoint & Tokenizer
TOKENIZER_PATH = "/kaggle/input/datasets/brindafadadu/cs781-phase1-ckpts/custom_bpe.json"
CHECKPOINT_PATH = "/kaggle/input/datasets/brindafadadu/cs781-phase1-ckpts/decoder_lm.pt"

# Phase 2 Hyperparameters
MAX_SEQ_LEN = 512
STRIDE = 128
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 3e-4
EPOCHS = 3
# 12 Official Labels
LABEL_NAMES = [
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE",
    "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
]
# BIO Scheme (25 tags total)
ID2LABEL = {0: "O"}
LABEL2ID = {"O": 0}
for idx, lbl in enumerate(LABEL_NAMES):
    ID2LABEL[1 + 2 * idx] = f"B-{lbl}"
    LABEL2ID[f"B-{lbl}"] = 1 + 2 * idx
    ID2LABEL[2 + 2 * idx] = f"I-{lbl}"
    LABEL2ID[f"I-{lbl}"] = 2 + 2 * idx
NUM_CLASSES = len(ID2LABEL)
IGNORE_INDEX = -100
print(f"Device: {DEVICE} | Classes: {NUM_CLASSES}")

Device: cuda | Classes: 25


In [59]:
SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]
END_OF_WORD = "</w>"

class ByteBPETokenizer:
    def __init__(self, vocab_size=8000):
        self.vocab_size = vocab_size
        self.merges = []
        self.merge_rank = {}
        self.itos = []
        self.stoi = {}

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls(vocab_size=data["vocab_size"])
        tok.itos = data["itos"]
        tok.stoi = {t: i for i, t in enumerate(tok.itos)}
        tok.merges = [tuple(p) for p in data["merges"]]
        tok.merge_rank = {p: i for i, p in enumerate(tok.merges)}
        return tok

    def _word_to_symbols(self, word):
        return list(word.encode("utf-8").decode("latin-1")) + [END_OF_WORD]

    def _apply_merges_to_word(self, word):
        symbols = self._word_to_symbols(word)
        while len(symbols) > 1:
            pairs_here = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
            candidates = [(self.merge_rank[p], idx) for idx, p in enumerate(pairs_here) if p in self.merge_rank]
            if not candidates:
                break
            _, idx = min(candidates)
            merged = symbols[idx] + symbols[idx + 1]
            symbols = symbols[:idx] + [merged] + symbols[idx + 2:]
        return symbols

    def encode_with_exact_offsets(self, text: str) -> Tuple[List[int], List[Tuple[int, int]]]:
        """
        Calculates EXACT character-level [start, end) offsets for every single subword token.
        """
        tokens = []
        offsets = []
        i = 0
        n = len(text)

        while i < n:
            while i < n and text[i].isspace():
                i += 1
            if i >= n:
                break
            word_start = i
            while i < n and not text[i].isspace():
                i += 1
            word_end = i
            word = text[word_start:word_end]

            subwords = self._apply_merges_to_word(word)
            curr_pos = word_start

            for sym in subwords:
                clean_sym = sym.replace(END_OF_WORD, "")
                try:
                    sym_chars = clean_sym.encode("latin-1").decode("utf-8", errors="replace")
                except Exception:
                    sym_chars = clean_sym

                sub_len = len(sym_chars)
                sub_start = curr_pos
                sub_end = min(word_end, curr_pos + sub_len)
                curr_pos = sub_end

                t_id = self.stoi.get(sym, self.stoi.get("<unk>", 1))
                tokens.append(t_id)
                offsets.append((sub_start, sub_end))

        return tokens, offsets

# Load tokenizer
tokenizer = ByteBPETokenizer.load(TOKENIZER_PATH)
print(f"Loaded tokenizer | Vocab size: {tokenizer.vocab_size}")

Loaded tokenizer | Vocab size: 8000


In [64]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def get_alibi_slopes(n_heads: int):
    def get_slopes_power_of_2(n):
        start = 2 ** (-(2 ** -(math.log2(n) - 3)))
        ratio = start
        return [start * (ratio ** i) for i in range(n)]

    if math.log2(n_heads).is_integer():
        return torch.tensor(get_slopes_power_of_2(n_heads))
    else:
        closest_pow2 = 2 ** math.floor(math.log2(n_heads))
        slopes_a = get_slopes_power_of_2(closest_pow2)
        slopes_b = get_slopes_power_of_2(2 * closest_pow2)[0::2][:n_heads - closest_pow2]
        return torch.tensor(slopes_a + slopes_b)

class LORMALinear(nn.Module):
    def __init__(self, original_linear: nn.Linear, rank: int = 16, alpha: float = 32.0):
        super().__init__()
        self.original_linear = original_linear
        self.rank = rank
        self.scaling = alpha / rank
        self.original_linear.weight.requires_grad = False

        in_f = original_linear.in_features
        out_f = original_linear.out_features
        self.lorma_down = nn.Linear(in_f, rank, bias=False)
        self.lorma_up = nn.Linear(rank, out_f, bias=False)

        nn.init.kaiming_uniform_(self.lorma_down.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lorma_up.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.original_linear(x) + self.lorma_up(self.lorma_down(x)) * self.scaling

class ALiBiBidirectionalAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.dropout = dropout

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.q_norm = RMSNorm(self.head_dim)
        self.k_norm = RMSNorm(self.head_dim)

        slopes = get_alibi_slopes(n_heads).unsqueeze(1).unsqueeze(2)
        self.register_buffer("slopes", slopes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, S, _ = x.shape
        q = self.q_proj(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)

        q = self.q_norm(q)
        k = self.k_norm(k)

        # Symmetric bidirectional distance
        pos = torch.arange(S, device=x.device)
        dist = -torch.abs(pos.unsqueeze(0) - pos.unsqueeze(1)).float()
        alibi_bias = self.slopes * dist.unsqueeze(0)

        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=alibi_bias,
            dropout_p=self.dropout if self.training else 0.0
        )
        out = out.transpose(1, 2).contiguous().view(B, S, self.d_model)
        return self.out_proj(out)

class SwiGLU(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj   = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn  = ALiBiBidirectionalAttention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn   = SwiGLU(d_model, d_ff, dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class Arch4LNERModel(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 640, n_layers: int = 16,
                 n_heads: int = 8, d_ff: int = 2560, num_classes: int = NUM_CLASSES,
                 dropout: float = 0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.norm_f = RMSNorm(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        x = self.drop(self.tok_emb(input_ids))
        for block in self.blocks:
            x = block(x)
        x = self.norm_f(x)
        return self.classifier(x)

    def apply_lorma(self, rank: int = 16, alpha: float = 32.0, device=DEVICE):
        for param in self.parameters():
            param.requires_grad = False

        for block in self.blocks:
            block.attn.q_proj = LORMALinear(block.attn.q_proj, rank=rank, alpha=alpha).to(device)
            block.attn.v_proj = LORMALinear(block.attn.v_proj, rank=rank, alpha=alpha).to(device)
            block.ffn.gate_proj = LORMALinear(block.ffn.gate_proj, rank=rank, alpha=alpha).to(device)
            block.ffn.up_proj = LORMALinear(block.ffn.up_proj, rank=rank, alpha=alpha).to(device)

        for param in self.classifier.parameters():
            param.requires_grad = True

# Calculate class weights: downweight 'O' class (index 0) so model aggressively detects entities
class_weights = torch.ones(NUM_CLASSES, device=DEVICE)
class_weights[0] = 0.25  # Downweight 'O' to 25% to combat severe class imbalance

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, ignore_index=-100):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.ignore_index = ignore_index

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce = F.cross_entropy(logits, targets, reduction="none", ignore_index=self.ignore_index, weight=self.alpha)
        pt = torch.exp(-ce)
        focal_loss = ((1.0 - pt) ** self.gamma) * ce
        mask = targets != self.ignore_index
        return focal_loss[mask].mean()

# Instantiate & Load Checkpoint
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
state_dict = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
d_model_ckpt = state_dict["tok_emb.weight"].shape[1]
vocab_size_ckpt = state_dict["tok_emb.weight"].shape[0]
block_indices = [int(k.split(".")[1]) for k in state_dict.keys() if k.startswith("blocks.") and k.split(".")[1].isdigit()]
n_layers_ckpt = max(block_indices) + 1 if block_indices else 16
n_heads = 8 if d_model_ckpt == 640 else 12

model = Arch4LNERModel(vocab_size=vocab_size_ckpt, d_model=d_model_ckpt, n_layers=n_layers_ckpt, n_heads=n_heads).to(DEVICE)
backbone_dict = {k: v for k, v in state_dict.items() if not k.startswith("lm_head") and not k.startswith("head")}
model.load_state_dict(backbone_dict, strict=False)

model.apply_lorma(rank=16, alpha=32.0, device=DEVICE)
model.to(DEVICE)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Params: {total_params:,} | Trainable Params: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")

Total Params: 112,722,585 | Trainable Params: 2,721,305 (2.41%)


In [61]:
def download_phase2_file(filename: str) -> str:
    return hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN
    )

print("Downloading dataset files...")
f1_text_path = download_phase2_file("Phase_2_LNER/Train/fold1-text.json")
f1_gold_path = download_phase2_file("Phase_2_LNER/Train/fold1-gold.json")
f2_text_path = download_phase2_file("Phase_2_LNER/Train/fold2-text.json")
f2_gold_path = download_phase2_file("Phase_2_LNER/Train/fold2-gold.json")
f3_text_path = download_phase2_file("Phase_2_LNER/Test/fold3-text.json")

with open(f1_text_path, "r", encoding="utf-8") as f: train_texts = json.load(f)
with open(f1_gold_path, "r", encoding="utf-8") as f: train_golds = json.load(f)
with open(f2_text_path, "r", encoding="utf-8") as f: val_texts = json.load(f)
with open(f2_gold_path, "r", encoding="utf-8") as f: val_golds = json.load(f)
with open(f3_text_path, "r", encoding="utf-8") as f: test_texts = json.load(f)

# Combine folds 1 and 2 to maximize training supervision
all_train_texts = {**train_texts, **val_texts}
all_train_golds = {**train_golds, **val_golds}

print(f"Total Combined Training Documents: {len(all_train_texts)}")

class ExactLNERDataset(Dataset):
    def __init__(self, docs: Dict[str, str], golds: Dict[str, List[Any]], tokenizer: ByteBPETokenizer,
                 max_len: int = MAX_SEQ_LEN, stride: int = STRIDE):
        self.samples = []
        for doc_id, text in docs.items():
            token_ids, offsets = tokenizer.encode_with_exact_offsets(text)

            token_labels = [0] * len(token_ids)
            if doc_id in golds:
                for annot in golds[doc_id]:
                    g_start, g_end, g_lbl = annot[0], annot[1], annot[2]
                    first_matched = False
                    for idx, (t_s, t_e) in enumerate(offsets):
                        if max(t_s, g_start) < min(t_e, g_end):
                            if not first_matched:
                                token_labels[idx] = LABEL2ID.get(f"B-{g_lbl}", 0)
                                first_matched = True
                            else:
                                token_labels[idx] = LABEL2ID.get(f"I-{g_lbl}", 0)

            for start_idx in range(0, max(1, len(token_ids)), max_len - stride):
                end_idx = min(len(token_ids), start_idx + max_len)
                c_ids = token_ids[start_idx:end_idx]
                c_lbls = token_labels[start_idx:end_idx]

                pad_len = max_len - len(c_ids)
                padded_ids = c_ids + [0] * pad_len
                padded_lbls = c_lbls + [IGNORE_INDEX] * pad_len

                self.samples.append({
                    "input_ids": torch.tensor(padded_ids, dtype=torch.long),
                    "labels": torch.tensor(padded_lbls, dtype=torch.long)
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

train_ds = ExactLNERDataset(all_train_texts, all_train_golds, tokenizer)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
print(f"Total training sliding windows: {len(train_ds)}")

Total training sliding windows: 1338


In [67]:
# Standard Cross-Entropy with Cosine Annealing (prevents false positive explosion)
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=3e-4, weight_decay=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

EPOCHS = 4
total_steps = (len(train_loader) // GRAD_ACCUM_STEPS) * EPOCHS
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-5)

print(f"Training {EPOCHS} epochs with balanced cross-entropy...")
model.train()
for epoch in range(EPOCHS):
    running_loss = 0.0
    optimizer.zero_grad()
    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(input_ids)
            loss = criterion(logits.view(-1, NUM_CLASSES), labels.view(-1)) / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        running_loss += loss.item() * GRAD_ACCUM_STEPS

        if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        if (step + 1) % 50 == 0:
            print(f"Epoch [{epoch + 1}/{EPOCHS}] | Step [{step + 1}/{len(train_loader)}] | Loss: {running_loss / 50:.4f}")
            running_loss = 0.0

print("✓ Training finished!")

Training 4 epochs with balanced cross-entropy...
Epoch [1/4] | Step [50/335] | Loss: 0.1903
Epoch [1/4] | Step [100/335] | Loss: 0.1673
Epoch [1/4] | Step [150/335] | Loss: 0.1591
Epoch [1/4] | Step [200/335] | Loss: 0.1429
Epoch [1/4] | Step [250/335] | Loss: 0.1386
Epoch [1/4] | Step [300/335] | Loss: 0.1477
Epoch [2/4] | Step [50/335] | Loss: 0.1161
Epoch [2/4] | Step [100/335] | Loss: 0.1221
Epoch [2/4] | Step [150/335] | Loss: 0.1307
Epoch [2/4] | Step [200/335] | Loss: 0.1248
Epoch [2/4] | Step [250/335] | Loss: 0.1289
Epoch [2/4] | Step [300/335] | Loss: 0.1284
Epoch [3/4] | Step [50/335] | Loss: 0.1113
Epoch [3/4] | Step [100/335] | Loss: 0.1084
Epoch [3/4] | Step [150/335] | Loss: 0.1018
Epoch [3/4] | Step [200/335] | Loss: 0.1031
Epoch [3/4] | Step [250/335] | Loss: 0.1057
Epoch [3/4] | Step [300/335] | Loss: 0.1075
Epoch [4/4] | Step [50/335] | Loss: 0.1055
Epoch [4/4] | Step [100/335] | Loss: 0.0931
Epoch [4/4] | Step [150/335] | Loss: 0.0925
Epoch [4/4] | Step [200/335] | 

In [71]:
STRIP_PUNCT = string.punctuation + " \t\n\r"

def clean_span(text: str, start: int, end: int, label: str):
    while start < end and text[start] in STRIP_PUNCT:
        start += 1
    while end > start and text[end - 1] in STRIP_PUNCT:
        end -= 1
    return start, end, label

def decode_spans_exact(doc_text: str, pred_ids: List[int], offsets: List[Tuple[int, int]]) -> List[List[Any]]:
    spans = []
    curr_label = None
    curr_start = -1
    curr_end = -1

    for t_id, (s_c, e_c) in zip(pred_ids, offsets):
        tag = ID2LABEL.get(t_id, "O")

        if tag.startswith("B-"):
            if curr_label is not None:
                cs, ce, cl = clean_span(doc_text, curr_start, curr_end, curr_label)
                if ce > cs: spans.append([cs, ce, cl])
            curr_label = tag[2:]
            curr_start = s_c
            curr_end = e_c
        elif tag.startswith("I-"):
            tag_lbl = tag[2:]
            if curr_label == tag_lbl:
                curr_end = e_c
            else:
                if curr_label is not None:
                    cs, ce, cl = clean_span(doc_text, curr_start, curr_end, curr_label)
                    if ce > cs: spans.append([cs, ce, cl])
                curr_label = tag_lbl
                curr_start = s_c
                curr_end = e_c
        else:
            if curr_label is not None:
                cs, ce, cl = clean_span(doc_text, curr_start, curr_end, curr_label)
                if ce > cs: spans.append([cs, ce, cl])
                curr_label = None

    if curr_label is not None:
        cs, ce, cl = clean_span(doc_text, curr_start, curr_end, curr_label)
        if ce > cs: spans.append([cs, ce, cl])

    unique_spans = []
    for sp in spans:
        if not any(sp[0] == u[0] and sp[1] == u[1] and sp[2] == u[2] for u in unique_spans):
            unique_spans.append(sp)

    return unique_spans

# Generate predictions with confidence threshold to kill false positives
model.eval()
test_predictions = {doc_id: [] for doc_id in test_texts.keys()}
print(f"Generating test predictions for {len(test_texts)} documents using exact argmax decoding...")
with torch.no_grad():
    for doc_id, text in test_texts.items():
        token_ids, offsets = tokenizer.encode_with_exact_offsets(text)
        if not token_ids:
            continue
        doc_spans = []
        for start_idx in range(0, max(1, len(token_ids)), MAX_SEQ_LEN - STRIDE):
            end_idx = min(len(token_ids), start_idx + MAX_SEQ_LEN)
            chunk_ids = token_ids[start_idx:end_idx]
            chunk_offsets = offsets[start_idx:end_idx]
            input_tensor = torch.tensor([chunk_ids], dtype=torch.long, device=DEVICE)
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                logits = model(input_tensor)
            # PURE ARGMAX (Exact same as local evaluation)
            preds = torch.argmax(logits[0], dim=-1).cpu().tolist()
            spans = decode_spans_exact(text, preds, chunk_offsets)
            doc_spans.extend(spans)
        # De-duplicate identical overlapping spans
        final_spans = []
        for sp in doc_spans:
            if not any(sp[0] == s[0] and sp[1] == s[1] and sp[2] == s[2] for s in final_spans):
                final_spans.append([int(sp[0]), int(sp[1]), str(sp[2])])
        test_predictions[doc_id] = final_spans
# Package into submission.zip
output_json = "/kaggle/working/prediction.json"
output_zip = "/kaggle/working/submission.zip"
with open(output_json, "w", encoding="utf-8") as f:
    f.write(json.dumps(test_predictions))
    f.flush()
    os.fsync(f.fileno())
if os.path.exists(output_zip):
    os.remove(output_zip)
with zipfile.ZipFile(output_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(output_json, arcname="prediction.json")
print(f"\n✓ Submission zip created: {output_zip} ({os.path.getsize(output_zip):,} bytes)")
# Sanity check
with zipfile.ZipFile(output_zip, "r") as zf:
    verified = json.loads(zf.read("prediction.json").decode("utf-8"))
    first_k = list(verified.keys())[0]
    print(f"✓ ZIP VERIFIED: Found {len(verified)} documents.")
    print(f"Sample prediction for doc {first_k}: {verified[first_k][:3]}")

Generating test predictions for 35 documents using exact argmax decoding...

✓ Submission zip created: /kaggle/working/submission.zip (38,830 bytes)
✓ ZIP VERIFIED: Found 35 documents.
Sample prediction for doc 1128835: [[86, 98, 'CASENO'], [145, 162, 'CASENO'], [172, 182, 'APP']]


In [69]:
# LOCAL STRICT EVALUATOR (EXACT CODABENCH PROTOCOL)

def evaluate_predictions_strictly(gold_data: Dict[str, List[Any]], pred_data: Dict[str, List[Any]]):
    """
    Computes Strict Macro-F1, Precision, and Recall across all 12 legal entity classes.
    An entity is strictly correct ONLY if [start, end, label] match exactly.
    """
    ALL_LABELS = [
        "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE",
        "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
    ]
    
    # Track True Positives, False Positives, False Negatives per label
    tp_per_label = {lbl: 0 for lbl in ALL_LABELS}
    fp_per_label = {lbl: 0 for lbl in ALL_LABELS}
    fn_per_label = {lbl: 0 for lbl in ALL_LABELS}

    for doc_id, gold_spans in gold_data.items():
        # Represent gold spans as a set of tuples: {(start, end, label), ...}
        gold_set = set()
        for sp in gold_spans:
            if len(sp) >= 3 and sp[2] in ALL_LABELS:
                gold_set.add((int(sp[0]), int(sp[1]), str(sp[2]).strip()))
                
        # Represent predicted spans as a set: {(start, end, label), ...}
        pred_set = set()
        if doc_id in pred_data:
            for sp in pred_data[doc_id]:
                if len(sp) >= 3 and sp[2] in ALL_LABELS:
                    pred_set.add((int(sp[0]), int(sp[1]), str(sp[2]).strip()))

        # Matches = Intersection
        matched = gold_set.intersection(pred_set)
        
        for sp in matched:
            tp_per_label[sp[2]] += 1
            
        for sp in pred_set - gold_set:
            fp_per_label[sp[2]] += 1
            
        for sp in gold_set - pred_set:
            fn_per_label[sp[2]] += 1

    # Calculate P, R, F1 per class and take the unweighted macro mean
    p_scores, r_scores, f1_scores = [], [], []

    print("\n" + "="*65)
    print(f"{'LABEL':<12} | {'GOLD':<6} | {'PRED':<6} | {'PREC':<8} | {'RECALL':<8} | {'F1':<8}")
    print("="*65)

    for lbl in ALL_LABELS:
        tp = tp_per_label[lbl]
        fp = fp_per_label[lbl]
        fn = fn_per_label[lbl]
        
        total_gold = tp + fn
        total_pred = tp + fp

        p = tp / total_pred if total_pred > 0 else 0.0
        r = tp / total_gold if total_gold > 0 else 0.0
        f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0

        p_scores.append(p)
        r_scores.append(r)
        f1_scores.append(f1)

        print(f"{lbl:<12} | {total_gold:<6} | {total_pred:<6} | {p:.4f}   | {r:.4f}   | {f1:.4f}")

    macro_p = sum(p_scores) / len(p_scores)
    macro_r = sum(r_scores) / len(r_scores)
    macro_f1 = sum(f1_scores) / len(f1_scores)

    print("="*65)
    print(f"Strict Macro-Precision: {macro_p:.6f}")
    print(f"Strict Macro-Recall:    {macro_r:.6f}")
    print(f"Strict Macro-F1:        {macro_f1:.6f}")
    print("="*65)

    return macro_f1, macro_p, macro_r

In [70]:
model.eval()
val_predictions = {}

print("Evaluating locally on Fold 2...")
with torch.no_grad():
    for doc_id, text in val_texts.items():
        token_ids, offsets = tokenizer.encode_with_exact_offsets(text)
        if not token_ids:
            val_predictions[doc_id] = []
            continue

        doc_spans = []
        for start_idx in range(0, max(1, len(token_ids)), MAX_SEQ_LEN - STRIDE):
            end_idx = min(len(token_ids), start_idx + MAX_SEQ_LEN)
            chunk_ids = token_ids[start_idx:end_idx]
            chunk_offsets = offsets[start_idx:end_idx]

            input_tensor = torch.tensor([chunk_ids], dtype=torch.long, device=DEVICE)
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                logits = model(input_tensor)
            preds = torch.argmax(logits[0], dim=-1).cpu().tolist()

            spans = decode_spans_exact(text, preds, chunk_offsets)
            doc_spans.extend(spans)

        unique_spans = []
        for sp in doc_spans:
            if not any(sp[0] == s[0] and sp[1] == s[1] and sp[2] == s[2] for s in unique_spans):
                unique_spans.append([int(sp[0]), int(sp[1]), str(sp[2])])

        val_predictions[doc_id] = unique_spans

# Run the local strict scoring!
macro_f1, macro_p, macro_r = evaluate_predictions_strictly(val_golds, val_predictions)

Evaluating locally on Fold 2...

LABEL        | GOLD   | PRED   | PREC     | RECALL   | F1      
APP          | 200    | 301    | 0.2691   | 0.4050   | 0.3234
RESP         | 190    | 278    | 0.2878   | 0.4211   | 0.3419
A.COUNSEL    | 125    | 202    | 0.2079   | 0.3360   | 0.2569
R.COUNSEL    | 97     | 158    | 0.2215   | 0.3608   | 0.2745
JUDGE        | 122    | 172    | 0.1047   | 0.1475   | 0.1224
WIT          | 120    | 200    | 0.3750   | 0.6250   | 0.4688
AUTH         | 468    | 711    | 0.4149   | 0.6303   | 0.5004
COURT        | 558    | 748    | 0.5829   | 0.7814   | 0.6677
STAT         | 735    | 1204   | 0.4012   | 0.6571   | 0.4982
PREC         | 559    | 1025   | 0.1112   | 0.2039   | 0.1439
DATE         | 762    | 993    | 0.6405   | 0.8346   | 0.7248
CASENO       | 337    | 557    | 0.2352   | 0.3887   | 0.2931
Strict Macro-Precision: 0.320985
Strict Macro-Recall:    0.482631
Strict Macro-F1:        0.384660
